<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/PINN_DeepXDE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!pip install deepxde

## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation $p_t = \kappa p_{xx}$ for $x \in [0, \pi]$, $t \in [0, 1]$, with initial condition $p(x, 0) = \sin(x)$ and boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [32]:
import deepxde as dde
import numpy as np
from deepxde.backend import tf

In [33]:
kappa = 1.0

In [34]:
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [35]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [36]:
def func(x):
    xx = x[:, 0]
    tt = x[:, 1]
    return (np.sin(xx) * np.exp(-tt))[:, None]

In [37]:
data = dde.data.TimePDE(
    geomtime,
    pde,
    [],
    num_domain=100,
    solution=func
)

In [38]:
layer_size = [2] + [20] * 2 + [1]
activation = "tanh"
initializer = "Glorot uniform"

In [39]:
net = dde.nn.FNN(layer_size, activation, initializer)

In [40]:
def output_transform(x, y):
    x_col = x[:, 0:1] # столбец x, форма (N, 1)
    t_col = x[:, 1:2] # столбец t, форма (N, 1)
    return tf.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

net.apply_output_transform(output_transform)

In [41]:
model = dde.Model(data, net)
model.compile("adam", lr=1e-3, metrics=["l2 relative error"])

Compiling model...
'compile' took 0.006469 s



In [42]:
resampler = dde.callbacks.PDEPointResampler(period=1)

In [43]:
losshistory, train_state = model.train(iterations=1000, callbacks=[resampler], display_every=50)

Training model...

Step      Train loss    Test loss     Test metric   
0         [2.74e+00]    [2.74e+00]    [9.36e-01]    
50        [6.30e-02]    [6.30e-02]    [1.92e-01]    
100       [2.10e-02]    [2.10e-02]    [8.70e-02]    
150       [6.72e-03]    [6.72e-03]    [4.37e-02]    
200       [3.16e-03]    [3.16e-03]    [2.15e-02]    
250       [2.33e-03]    [2.33e-03]    [1.49e-02]    
300       [1.92e-03]    [1.92e-03]    [1.33e-02]    
350       [1.63e-03]    [1.63e-03]    [1.24e-02]    
400       [1.40e-03]    [1.40e-03]    [1.16e-02]    
450       [1.21e-03]    [1.21e-03]    [1.08e-02]    
500       [1.06e-03]    [1.06e-03]    [1.01e-02]    
550       [9.34e-04]    [9.34e-04]    [9.41e-03]    
600       [8.30e-04]    [8.30e-04]    [8.79e-03]    
650       [7.43e-04]    [7.43e-04]    [8.23e-03]    
700       [6.71e-04]    [6.71e-04]    [7.72e-03]    
750       [6.09e-04]    [6.09e-04]    [7.28e-03]    
800       [5.56e-04]    [5.56e-04]    [6.88e-03]    
850       [5.10e-04]    [5.

In [44]:
import plotly.express as px
import pandas as pd
import numpy as np

In [45]:
domain_x = [0, np.pi]
domain_t = [0, 1]

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
    x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
    t_plot = t_val * np.ones_like(x_plot)
    inputs_plot = np.concatenate([x_plot, t_plot], axis=1)

    p_plot = model.predict(inputs_plot)

    p_exact = np.exp(-t_val) * np.sin(x_plot)
    df_temp = pd.DataFrame({
        'x': x_plot.flatten(),
        'PINN': p_plot.flatten(),
        'Exact': p_exact.flatten(),
        'Time': t_val
    })
    plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')

fig = px.line(df_melted, x='x', y='p', animation_frame='Time',
              color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(xaxis_title='x', yaxis_title='p(x, t)')
fig.show()

In [96]:
t_fixed = 0.3
x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
t_plot = t_fixed * np.ones_like(x_plot)
inputs_plot = np.concatenate([x_plot, t_plot], axis=1)

p_plot = model.predict(inputs_plot)
p_star = np.exp(-t_fixed) * np.sin(x_plot)
norm = np.mean(np.abs(p_star))
eps_point = np.abs(p_plot - p_star) / norm

df_err = pd.DataFrame({'x': x_plot.flatten(), 'rel_error': eps_point.flatten()})
fig = px.line(df_err, x='x', y='rel_error', title=f'Ошибка при t={t_fixed}')
fig.update_xaxes(range=[domain_x[0], domain_x[1]])
fig.show()

## Loss:

In [46]:
import plotly.graph_objects as go
import numpy as np

In [47]:
losses = np.array(losshistory.loss_train).flatten()
steps = np.array(losshistory.steps)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=losses, mode='lines+markers', name='Dirichlet'))
fig.update_layout(
    title='Сходимость обучения PINN (DeepXDE)',
    xaxis_title='Итерация',
    yaxis_title='Функция потерь',
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

## Что будет если менять число эпох?

In [48]:
import deepxde.nn.initializers as dde_initializers

In [49]:
x = np.linspace(0, np.pi, 200)
t = np.linspace(0, 1, 100)
X, T = np.meshgrid(x, t)
inputs = np.stack([X.ravel(), T.ravel()], axis=1)
p_exact = np.exp(-T) * np.sin(X)

In [50]:
checkpoints = [1000, 2000, 3000, 4000, 5000]
results = np.zeros((5, len(checkpoints)))

In [51]:
for i, seed in enumerate(range(1, 6)):
    print(f"Обучение №{seed}")
    dde_initializers.INITIALIZER_DICT["Glorot uniform"] = tf.keras.initializers.GlorotUniform(seed=seed)
    dde.config.set_random_seed(seed)

    data = dde.data.TimePDE(geomtime, pde, [], num_domain=100, solution=func)
    net = dde.nn.FNN(layer_size, activation, initializer)
    net.apply_output_transform(output_transform)
    model = dde.Model(data, net)
    model.compile("adam", lr=1e-3)

    resampler = dde.callbacks.PDEPointResampler(period=1)

    for j, step in enumerate(checkpoints):
        iterations_to_train = 1000 if j == 0 else (checkpoints[j] - checkpoints[j-1])
        model.train(iterations=iterations_to_train, callbacks=[resampler], display_every=5000)

        p_pred = model.predict(inputs).reshape(X.shape)
        results[i, j] = np.linalg.norm(p_pred - p_exact) / np.linalg.norm(p_exact)

Обучение №1
Compiling model...
'compile' took 0.017894 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.29e-01]    [2.29e-01]    []  
1000      [2.52e-04]    [2.52e-04]    []  

Best model at step 1000:
  train loss: 2.52e-04
  test loss: 2.52e-04
  test metric: []

'train' took 21.002662 s

Training model...

Step      Train loss    Test loss     Test metric
1000      [2.52e-04]    [2.52e-04]    []  
2000      [4.78e-05]    [4.78e-05]    []  

Best model at step 2000:
  train loss: 4.78e-05
  test loss: 4.78e-05
  test metric: []

'train' took 13.394624 s

Training model...

Step      Train loss    Test loss     Test metric
2000      [4.78e-05]    [4.78e-05]    []  
3000      [2.30e-05]    [2.30e-05]    []  

Best model at step 3000:
  train loss: 2.30e-05
  test loss: 2.30e-05
  test metric: []

'train' took 13.290291 s

Training model...

Step      Train loss    Test loss     Test metric
3000      [2.30e-05]    [2.30e-05]    []  
4000      [1.88e-

In [52]:
mean_errors = np.mean(results, axis=0)
for step, error in zip(checkpoints, mean_errors):
    print(f"Шаг {step} -> Средняя L2 = {error:.4e}")

Шаг 1000 -> Средняя L2 = 6.6565e-03
Шаг 2000 -> Средняя L2 = 2.5075e-03
Шаг 3000 -> Средняя L2 = 1.6850e-03
Шаг 4000 -> Средняя L2 = 1.0172e-03
Шаг 5000 -> Средняя L2 = 3.0258e-03


In [53]:
fig = go.Figure()

for i in range(5):
    fig.add_trace(go.Scatter(
        x=checkpoints,
        y=results[i, :],
        mode='lines+markers',
        name=f'Обучение №{i+1}'
    ))

fig.add_trace(go.Scatter(
    x=checkpoints,
    y=mean_errors,
    mode='lines+markers',
    name='СРЕДНЕЕ ЗНАЧЕНИЕ',
    line=dict(color='black', width=4),
    marker=dict(size=10, symbol='diamond')
))

fig.update_layout(
    xaxis=dict(
        title='Количество итераций (эпох)',
        tickvals=checkpoints,
        gridcolor='lightgray'
    ),
    yaxis=dict(
        title='Относительная погрешность L2',
        exponentformat='e',
        gridcolor='lightgray'
    ),
    title=dict(
        text='Результаты 5 обучений PINN',
        x=0.5
    ),
    plot_bgcolor='white'
)

fig.show()

## Что будет если менять число точек коллокации?

In [54]:
domain_points_list = [50, 100, 200, 500]
points_results = np.zeros((5, len(domain_points_list)))

In [55]:
for i, seed in enumerate(range(1, 6)):
    print(f"Обучение №{seed}")
    dde_initializers.INITIALIZER_DICT["Glorot uniform"] = tf.keras.initializers.GlorotUniform(seed=seed)
    dde.config.set_random_seed(seed)

    for j, points in enumerate(domain_points_list):
        data = dde.data.TimePDE(geomtime, pde, [], num_domain=points, solution=func)

        net = dde.nn.FNN(layer_size, activation, initializer)
        net.apply_output_transform(output_transform)
        model = dde.Model(data, net)
        model.compile("adam", lr=1e-3)

        resampler = dde.callbacks.PDEPointResampler(period=1)

        model.train(iterations=3000, callbacks=[resampler], display_every=5000)

        p_pred = model.predict(inputs).reshape(X.shape)
        points_results[i, j] = np.linalg.norm(p_pred - p_exact) / np.linalg.norm(p_exact)


Обучение №1
Compiling model...
'compile' took 0.012273 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.32e-01]    [2.32e-01]    []  
3000      [1.93e-05]    [1.93e-05]    []  

Best model at step 3000:
  train loss: 1.93e-05
  test loss: 1.93e-05
  test metric: []

'train' took 40.468600 s

Compiling model...
'compile' took 0.004864 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.29e-01]    [2.29e-01]    []  
3000      [2.30e-05]    [2.30e-05]    []  

Best model at step 3000:
  train loss: 2.30e-05
  test loss: 2.30e-05
  test metric: []

'train' took 41.496301 s

Compiling model...
'compile' took 0.004364 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.29e-01]    [2.29e-01]    []  
3000      [2.73e-05]    [2.73e-05]    []  

Best model at step 3000:
  train loss: 2.73e-05
  test loss: 2.73e-05
  test metric: []

'train' took 42.375789 s

Compiling model...
'compile' took 0.0

In [56]:
mean_points_errors = np.mean(points_results, axis=0)
for points, error in zip(domain_points_list, mean_points_errors):
    print(f"Точки {points} -> Средняя L2 = {error:.4e}")

Точки 50 -> Средняя L2 = 1.1702e-03
Точки 100 -> Средняя L2 = 1.6850e-03
Точки 200 -> Средняя L2 = 1.5980e-03
Точки 500 -> Средняя L2 = 1.6346e-03


In [57]:
fig = go.Figure()

for i in range(5):
    fig.add_trace(go.Scatter(
        x=domain_points_list,
        y=points_results[i, :],
        mode='lines+markers',
        name=f'Обучение №{i+1}'
    ))

fig.add_trace(go.Scatter(
    x=domain_points_list,
    y=mean_points_errors,
    mode='lines+markers',
    name='СРЕДНЕЕ ЗНАЧЕНИЕ',
    line=dict(color='black', width=4),
    marker=dict(size=10, symbol='diamond')
))

fig.update_layout(
    xaxis=dict(
        title='Количество точек коллокации',
        tickvals=domain_points_list,
        gridcolor='lightgray'
    ),
    yaxis=dict(
        title='Относительная погрешность L2',
        exponentformat='e',
        gridcolor='lightgray'
    ),
    title=dict(
        text='Результаты 5 обучений PINN',
        x=0.5
    ),
    plot_bgcolor='white'
)

fig.show()

## Что будет если менять оптимизаторы?

In [58]:
optimizers_list = ["adam", "sgd", "nadam"]
optimizers_results = np.zeros((5, len(optimizers_list)))

In [59]:
for i, seed in enumerate(range(1, 6)):
    print(f"Обучение {seed}")
    dde_initializers.INITIALIZER_DICT["Glorot uniform"] = tf.keras.initializers.GlorotUniform(seed=seed)
    dde.config.set_random_seed(seed)

    for j, opt in enumerate(optimizers_list):
        print(f"Оптимизатор: {opt.upper()}...")

        data = dde.data.TimePDE(geomtime, pde, [], num_domain=100, solution=func)
        net = dde.nn.FNN(layer_size, activation, initializer)
        net.apply_output_transform(output_transform)
        model = dde.Model(data, net)

        model.compile(opt, lr=1e-3)

        resampler = dde.callbacks.PDEPointResampler(period=1)

        model.train(iterations=2000, callbacks=[resampler], display_every=5000)

        p_pred = model.predict(inputs).reshape(X.shape)
        optimizers_results[i, j] = np.linalg.norm(p_pred - p_exact) / np.linalg.norm(p_exact)


Обучение 1
Оптимизатор: ADAM...
Compiling model...
'compile' took 0.011730 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.29e-01]    [2.29e-01]    []  
2000      [4.78e-05]    [4.78e-05]    []  

Best model at step 2000:
  train loss: 4.78e-05
  test loss: 4.78e-05
  test metric: []

'train' took 27.854681 s

Оптимизатор: SGD...
Compiling model...
'compile' took 0.007565 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.29e-01]    [2.29e-01]    []  
2000      [1.84e-03]    [1.84e-03]    []  

Best model at step 2000:
  train loss: 1.84e-03
  test loss: 1.84e-03
  test metric: []

'train' took 26.649038 s

Оптимизатор: NADAM...
Compiling model...
'compile' took 0.006372 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.29e-01]    [2.29e-01]    []  
2000      [8.34e-05]    [8.34e-05]    []  

Best model at step 2000:
  train loss: 8.34e-05
  test loss: 8.34e-05
  test metric: []

'

In [60]:
mean_opt_errors = np.mean(optimizers_results, axis=0)
for opt, error in zip(optimizers_list, mean_opt_errors):
    print(f"Оптимизатор {opt.upper()} -> Средняя L2 = {error:.4e}")

Оптимизатор ADAM -> Средняя L2 = 2.5075e-03
Оптимизатор SGD -> Средняя L2 = 1.2167e-02
Оптимизатор NADAM -> Средняя L2 = 2.3508e-03
